# 📚 Gerador de Dataset para Fine-Tuning (Instrução + Resposta)

Este notebook cria um dataset no formato `question` / `answer` a partir de documentos `.pdf` ou `.txt`.

**Objetivo:** gerar pares de pergunta curta + resposta direta com base no conteúdo do documento, usando um modelo de linguagem local (Qwen2.5-1.5B-Instruct).

**Saída:** arquivo JSONL (uma linha por exemplo), pronto para treinar ou ajustar um modelo RAG ou de QA.

---
## Etapas principais:
1. Instalar dependências necessárias
2. Extrair texto do arquivo (PDF ou TXT)
3. Dividir o texto em pequenos trechos (chunks)
4. Para cada trecho, usar um modelo Hugging Face para gerar uma pergunta e uma resposta curta
5. Salvar os pares gerados em formato JSONL

Vamos começar!


## 1. Instalação das bibliotecas

- `transformers`, `torch`, `accelerate`: para carregar e executar modelos da Hugging Face.
- `pdfplumber`: para extrair texto de PDFs.
- `tqdm`: para mostrar a barra de progresso.

> **Nota:** se você for usar apenas arquivos `.txt`, o `pdfplumber` não é obrigatório, mas vamos mantê-lo para suportar ambos os formatos.

In [ ]:
!pip install transformers torch accelerate pdfplumber tqdm -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 2.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.4/68.4 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 42.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 32.4 MB/s eta 0:00:00


## 2. Importações e configurações

Vamos importar os módulos necessários e desabilitar avisos desnecessários do Transformers para manter a saída limpa.

In [ ]:
import pdfplumber
import json
import torch
from transformers import pipeline, logging
from tqdm import tqdm

# Desativa avisos (apenas erros críticos serão mostrados)
logging.set_verbosity_error()

## 3. Função para extrair texto

Dependendo da extensão do arquivo, chamamos o método adequado:
- **PDF** → `pdfplumber`
- **TXT** → leitura simples com `open`

In [ ]:
def extract_text_from_file(file_path):
    """
    Extrai texto de um arquivo .pdf ou .txt.
    Retorna uma string com todo o conteúdo.
    """
    if file_path.lower().endswith('.pdf'):
        text = ""
        with pdfplumber.open(file_path) as pdf:
            for page in pdf.pages:
                page_text = page.extract_text()
                if page_text:
                    text += page_text + "\n"
        return text
    elif file_path.lower().endswith('.txt'):
        with open(file_path, 'r', encoding='utf-8') as f:
            return f.read()
    else:
        raise ValueError("Formato não suportado. Use .pdf ou .txt")


## 4. Divisão em trechos (chunks)

Modelos de linguagem têm um limite de tokens. Por isso dividimos o texto em blocos pequenos (ex.: 500 caracteres). Cada bloco será usado para gerar um par pergunta/resposta.

In [ ]:
def split_text(text, max_chunk_length=1200):
    """
    Divide o texto em chunks com base em quebras de linha.
    Cada chunk terá no máximo max_chunk_length caracteres.
    """
    paragraphs = text.split("\n")
    chunks = []
    current_chunk = ""

    for para in paragraphs:
        if len(current_chunk) + len(para) < max_chunk_length:
            current_chunk += para + "\n"
        else:
            if current_chunk:
                chunks.append(current_chunk.strip())
            current_chunk = para + "\n"

    if current_chunk:
        chunks.append(current_chunk.strip())

    return chunks


## 5. Gerar (Instrução, Resposta) a partir de um trecho

Usamos um modelo da família **Qwen2.5-1.5B-Instruct**, que é leve e muito bom para seguir instruções. O modelo recebe um prompt bem estruturado e deve retornar algo como:

```
INSTRUCTION: O que é X?
RESPONSE: Definição de X.
```

A função abaixo extrai esses campos e retorna a pergunta e a resposta.

In [ ]:
def generate_instruction_response(chunk, hf_pipeline):
    """
    Dado um chunk de texto, gera 3 pares de pergunta e resposta.
    """

    prompt = f"""
Você está criando um dataset para treinamento de um modelo de linguagem.

Com base apenas no texto abaixo, gere EXATAMENTE 3 pares de pergunta e resposta.

Regras:

- Use somente informações presentes no texto.
- Não invente informações.
- Não utilize conhecimento externo.
- As perguntas devem ser diferentes entre si.
- Priorize informações sobre:
  - segurança
  - operação
  - manutenção
  - transporte
  - armazenamento
  - regulagens
  - especificações técnicas
- As respostas devem ser objetivas e completas.

Formato obrigatório:

INSTRUCTION 1: <pergunta>
RESPONSE 1: <resposta>

INSTRUCTION 2: <pergunta>
RESPONSE 2: <resposta>

INSTRUCTION 3: <pergunta>
RESPONSE 3: <resposta>

TEXTO:
\"\"\"
{chunk}
\"\"\"
"""

    messages = [{"role": "user", "content": prompt}]

    try:
        outputs = hf_pipeline(
            messages,
            max_new_tokens=300,
            max_length=None,
            return_full_text=False
        )

        content = outputs[0]["generated_text"]

        pairs = []

        for i in range(1, 4):
            try:
                instruction = content.split(
                    f"INSTRUCTION {i}:"
                )[1].split(
                    f"RESPONSE {i}:"
                )[0].strip()

                if i < 3:
                    answer = content.split(
                        f"RESPONSE {i}:"
                    )[1].split(
                        f"INSTRUCTION {i+1}:"
                    )[0].strip()
                else:
                    answer = content.split(
                        f"RESPONSE {i}:"
                    )[1].strip()

                if len(instruction.split()) >= 3 and len(answer.split()) >= 3:
                    pairs.append((instruction, answer))

            except:
                pass

        return pairs

    except Exception:
        return []

## 6. Salvar dataset em formato JSONL

Cada linha do arquivo será um objeto `{"question": "...", "answer": "..."}`. Esse formato é amplamente usado para fine-tuning de modelos de QA.

In [ ]:
def save_to_jsonl(pairs, output_file="dataset.jsonl"):
    """
    pairs: lista de tuplas (instrução, resposta)
    output_file: nome do arquivo de saída
    """
    with open(output_file, "w", encoding="utf-8") as f:
        for instruction, answer in pairs:
            if instruction and answer:
                example = {
                    "Instruction": instruction,
                    "Output": answer
                }
                f.write(json.dumps(example, ensure_ascii=False) + "\n")


## 7. Função principal

A função `generate_dataset` coordena todo o fluxo:
- Carrega o modelo (baixa os pesos se necessário)
- Extrai texto do arquivo (PDF ou TXT)
- Divide em chunks
- Para cada chunk, chama o modelo e coleta os pares
- Salva o resultado

In [ ]:
def generate_dataset(file_path, model_id="Qwen/Qwen2.5-1.5B-Instruct",
                    output_file="dataset.jsonl", max_chunks=None):
    """
    file_path: caminho para o arquivo .pdf ou .txt
    model_id: identificador do modelo no Hugging Face Hub
    output_file: nome do arquivo de saída (JSONL)
    max_chunks: se especificado, limita a quantidade de chunks processados
    """
    print(f"🔄 Carregando modelo: {model_id} ...")

    # Pipeline para geração de texto
    hf_pipeline = pipeline(
        "text-generation",
        model=model_id,
        device_map="auto",        # usa GPU se disponível
        torch_dtype=torch.bfloat16 # reduz consumo de memória
    )

    print("📄 Extraindo texto do arquivo...")
    text = extract_text_from_file(file_path)

    print("✂️  Dividindo em chunks...")
    chunks = split_text(text)

    if max_chunks:
        chunks = chunks[:max_chunks]

    print(f"🧠 Gerando pares (instrução + resposta) para {len(chunks)} chunks...")
    all_generated_pairs = [] # Initialize a list to collect all pairs

    for chunk in tqdm(chunks, desc="Processando chunks"):
        if len(chunk.strip()) < 10:  # ignora chunks muito curtos
            continue

        generated_pairs_for_chunk = generate_instruction_response(chunk, hf_pipeline)

        if generated_pairs_for_chunk:
            all_generated_pairs.extend(generated_pairs_for_chunk)

    # After the loop, save all collected pairs
    save_to_jsonl(all_generated_pairs, output_file)
    print(f"\n✅ Dataset salvo em: {output_file} ({len(all_generated_pairs)} exemplos gerados)")


## 8. Execução

Agora basta informar o caminho do seu arquivo (PDF ou TXT) e, opcionalmente, o número máximo de chunks para testar.

> **Dica:** comece com `max_chunks=5` para verificar se a geração está funcionando. Depois remova o parâmetro ou aumente o valor para processar o documento inteiro.

O modelo será baixado na primeira execução (cerca de 3 GB). Aguarde o download completo.

In [ ]:
import os

print(os.listdir())

['.config', 'sample_data']


In [ ]:
from google.colab import files

uploaded = files.upload()

Saving 20260062399 (1).pdf to 20260062399 (1).pdf


In [ ]:
caminho_arquivo = "20260062399 (1).pdf"

In [ ]:
generate_dataset(
    file_path="20260062399 (1).pdf",
    model_id="Qwen/Qwen2.5-1.5B-Instruct",
    output_file="dataset_gerado_v3.jsonl",
    max_chunks=None
)

🔄 Carregando modelo: Qwen/Qwen2.5-1.5B-Instruct ...


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

📄 Extraindo texto do arquivo...
✂️  Dividindo em chunks...
🧠 Gerando pares (instrução + resposta) para 51 chunks...


Processando chunks: 100%|██████████| 51/51 [08:43<00:00, 10.26s/it]


✅ Dataset salvo em: dataset_gerado_v3.jsonl (152 exemplos gerados)


In [ ]:
import os

print(os.path.exists("dataset_gerado_v3.jsonl"))

True


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import shutil

shutil.copy(
    "dataset_gerado_v3.jsonl",
    "/content/drive/MyDrive/dataset_gerado_v3.jsonl"
)

'/content/drive/MyDrive/dataset_gerado_v3.jsonl'

In [ ]:
import os

print(
    os.path.exists(
        "/content/drive/MyDrive/dataset_gerado_v3.jsonl"
    )
)

True


In [ ]:
with open("dataset_gerado_v3.jsonl", "r", encoding="utf-8") as f:
    for i in range(20):
        print(f.readline())

{"Instruction": "Qual é a principal função do manual de instruções fornecido pela Marchesan S.A.?", "Output": "O manual de instruções serve para orientar sobre como usar efetivamente o equipamento, incluindo suas funções, procedimentos de operação e manutenção, além de garantir uma vida útil e confiabilidade máxima."}

{"Instruction": "Em caso de problemas de segurança, onde estão localizadas as instruções específicas?", "Output": "Os avisos de segurança encontram-se em diversas partes do manual, desde a introdução até a última página, sempre destacados com letras mais grossas e/ou coloridos para facilmente identificar."}

{"Instruction": "Como o usuário pode obter assistência técnica sobre a operação do equipamento?", "Output": "Para obter assistência técnica, você pode entrar em contato diretamente com a equipe de revendedores ou técnicos da Marchesan S.A., que estarão disponíveis para responder qualquer dúvida ou esclarecer dúvidas sobre a operação do equipamento."}

{"Instruction":

In [ ]:
with open("dataset_gerado_v3.jsonl", "r", encoding="utf-8") as f:
    linhas = f.readlines()

print("Quantidade de exemplos:", len(linhas))

Quantidade de exemplos: 152


## 🔍 Observações finais

- O modelo utilizado é o **Qwen2.5-1.5B-Instruct**, que funciona bem em CPU ou GPU.
- Para documentos longos, o processo pode levar alguns minutos/horas, dependendo do hardware.
- O dataset gerado pode ser usado para fine-tuning de modelos menores (como BERT, Llama, etc.) ou como base para RAG.
- Se desejar modificar o comprimento das perguntas/respostas, ajuste o `prompt` na função `generate_instruction_response`.

**Divirta-se gerando dados!** 🚀